# LabOne Q — Lecture 4: Automation

This is a stretch lecture: automating a full calibration pipeline end to end, following the structure of the
LabOne Q User Manual:

1. **The Automation framework** — [Automation](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07a_automation/tutorials/00_automation.html):
   layers, nodes, dependencies, status, and retry logic — a small, generic, hardware-free example
2. **`WorkflowAutomation`** — [Experiment Workflow Automation](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/sources/experiment_workflow_automation.html):
   the same framework, specialised to run LabOne Q experiment workflows
3. **Result injection** — running a full pipeline without hardware, using pre-recorded results
4. **A real example** — a simple single-qubit tune-up pipeline (Amplitude Rabi → Ramsey → Amplitude Fine →
   DRAG → Amplitude Fine → Single-Qubit RB) automated end to end for three qubits of a six-qubit chip,
   including what happens — and how to recover automatically — when a qubit fails evaluation

Everything below runs in **emulation mode**, with pre-recorded results injected at the measurement step
(section 3), so the whole pipeline — compilation, analysis, evaluation and the QPU parameter updates — is
real and reproducible without an instrument rack. The recorded results are *synthetic*: they are computed
from a small physical model of each qubit (see `automation_helpers/generate_synthetic_data.py`), with
1024-shot statistics and readout noise, and the chip's stored calibration is deliberately a little off
from that model — so the pipeline has something to correct.

## 1. The Automation framework

The [`laboneq.automation`](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/07a_automation/tutorials/00_automation.html)
package is a generic, graph-based execution framework. It organises a suite of routines into **layers**,
where each layer applies a function to a set of **nodes** — the minimal resources the function acts on
(here: a qubit; in general, anything). Layers are run in the order they were added, and each layer can
**depend on** an earlier one.

To use it, subclass `AutomationLayer` and implement `run_executable_core`: given the current layer's
nodes and parameters, run the function and decide, per node, whether it succeeded.

In [ ]:
import random

import attrs

from laboneq.automation import Automation, AutomationLayer, AutomationLayerResult
from laboneq.automation.logic import FixedParameterUpdate
from laboneq.core.utilities.dsl_dataclass_decorator import classformatter


def generate_random_ints(node_keys, parameters):
    """A stand-in for a real experiment: one seeded random integer per node."""
    return {k: random.Random(parameters[k]["seed"]).randint(1, 10) for k in node_keys}


@classformatter
@attrs.define
class ExampleLayer(AutomationLayer):
    def run_executable_core(self, automation):
        results = self.function(self.target_node_keys, self.target_parameters)
        # "pass" if the four random numbers add up to more than 20
        total = sum(results.values())
        successes = {node_key: total > 20 for node_key in results}
        return AutomationLayerResult(results=results, successes=successes)

Build a one-layer graph with four nodes, and run it. Each node's parameters (its random seed) are
supplied through the `Automation` object's `parameters` dict, keyed first by layer, then by node:

In [ ]:
auto_params = {
    "l1": {"n1": {"seed": 1}, "n2": {"seed": 2}, "n3": {"seed": 3}, "n4": {"seed": 4}},
}

auto = Automation(name="toy", parameters=auto_params)

l1 = ExampleLayer(
    generate_random_ints, ["n1", "n2", "n3", "n4"], key="l1", depends_on={"root"},
)
auto.add_layer(l1)

print("status before running:", auto["l1"].status)
auto.run()
print("status after running: ", auto["l1"].status)
print("results:", auto["l1"].results)

With these particular seeds the four numbers add up to less than 20, so the layer **fails** — that is
expected, and is exactly the situation `logic` exists for.

### Adding retry logic

`AutomationLogic` subclasses let a layer automatically retry with adjusted parameters when it fails.
`FixedParameterUpdate` is the simplest one: on every failure, apply the same fixed change and try again,
until it passes (or a maximum number of failures is reached).

In [ ]:
auto.reset()  # clear statuses and results, keep the graph structure

auto["l1"].logic = FixedParameterUpdate(
    new_layer_key="l1",
    parameter_changes={
        "n1": {"seed": 1}, "n2": {"seed": 1}, "n3": {"seed": 1}, "n4": {"seed": 1},
    },
)

auto.run()
print("status:     ", auto["l1"].status)
print("fail_count: ", auto["l1"].fail_count)
print("pass_count: ", auto["l1"].pass_count)

The layer now passes — the seeds were incremented automatically after each failed attempt until the
sum exceeded 20. `fail_count` / `pass_count` show exactly how many attempts that took, per node.

The generic framework also gives a **live web viewer** (`laboneq.automation.web_viewer.start_web_viewer`)
and a static graph plot (`auto.plot()`) — both are shown in full in section 4, once we have something more
interesting to look at than random integers.

## 2. `WorkflowAutomation` — automating experiment workflows

`laboneq_applications.automation` specialises the generic framework for LabOne Q: `WorkflowLayer` is an
`AutomationLayer` whose "function" is an experiment workflow (like `amplitude_rabi.experiment_workflow`
from Lecture 3) and whose nodes are qubits. Running a `WorkflowLayer` builds, compiles, runs and analyses
the experiment for every qubit in the layer — success/failure comes straight from the workflow's own
`evaluate_experiment` task.

We reuse the two-qubit demo platform from Lecture 2/3 and the `qubit_spectroscopy` experiment for a quick,
self-contained illustration.

In [ ]:
import numpy as np
from laboneq.simple import *

from laboneq_applications.automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import qubit_spectroscopy
from laboneq_applications.qpu_types.tunable_transmon.demo_qpus import demo_platform

qs_platform = demo_platform(n_qubits=2)
qs_setup = qs_platform.setup
qs_qpu = qs_platform.qpu
qs_session = Session(qs_setup)
qs_session.connect(do_emulation=True)

qs_auto_params = {
    "qs1": {
        "workflow_parameters": {
            "q0": {"frequencies": np.linspace(6.1e9, 6.6e9, 101)},
            "q1": {"frequencies": np.linspace(6.1e9, 6.6e9, 101)},
        },
        "options": {"evaluate": True, "update": True, "count": 1024},
    }
}

qs_auto = WorkflowAutomation(qs_session, qs_qpu, name="qs_demo", parameters=qs_auto_params)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow, ["q0", "q1"], key="qs1", depends_on={"root"},
)
qs_auto.add_layer(qs1)

qs_auto.run()
print("status:", qs_auto["qs1"].status)

### Grabbing a single task's output for inspection

Every `WorkflowLayer` keeps the full `WorkflowResult` for each set of qubits it ran, under
`workflow_results[qubit_tuple]` — exactly the same `result.tasks[...]` access from Lecture 3, just one
level deeper (layer → qubits → workflow → task):

In [ ]:
qs1_result = qs_auto["qs1"].workflow_results[("q0", "q1")]

fit_result = qs1_result.tasks["analysis_workflow"].tasks["fit_data"].output["q0"]
print("fit r-squared for q0:", fit_result.rsquared)

evaluation = qs1_result.tasks["evaluate_experiment"].output["q0"]
print("evaluation for q0:   ", evaluation)

`evaluation["success"]` is what drove the node's pass/fail status above; `evaluation["update"]` says
whether the fitted frequency moved far enough from the old value to be worth writing back to the QPU (it
didn't here, so `qs_qpu["q0"].parameters.resonance_frequency_ge` is unchanged).

## 3. Result injection

Everything above ran against `demo_platform`'s emulator, which returns essentially noise-free synthetic
data — fine for a quick illustration, but not representative of what a real bring-up run looks like.

The `run_experiment` task (used by every experiment workflow) accepts an `inject_results` option:

```python
options.inject_results({
    "paths": ["saved_results/rabi_results.json"],
    "use": "emulation",       # "no" | "emulation" | "always"
    "check_shapes": False,
})
```

When set, `run_experiment` still runs the experiment (compiling and, if connected, running it) but then
**replaces** the returned `Results` with a previously saved `Results` object loaded from `paths` — as long
as `use` allows it for the current connection mode. `"emulation"` (the default) injects only when the
session is running in emulation; `"always"` injects regardless (useful for regression-testing analysis
code against fixed data); `"no"` disables injection entirely.

**One rule to know**: injection matches by *acquisition handle*, not by shape. Every handle produced by
the real run (e.g. `"q1/result"`, `"q1/cal_trace/e"`) must also be present in the injected file — so the
injected file needs to have been recorded for (at least) the same qubits you are running with. Extra
qubits in the injected file are simply ignored. With `check_shapes: True` (the default), the *array
shapes* must match too; the demo below sets it to `False`.

Section 4 uses `inject_results` with one JSON file per calibration step, each holding a recorded
`Results` object for every qubit on the chip — generated once by
`automation_helpers/generate_synthetic_data.py` and committed with the course.

## 4. A simple 1Q tune-up for three qubits

We now put it all together: an automated, six-layer single-qubit calibration pipeline, run for three
qubits of a serialized six-qubit QPU, using injected results throughout so the entire pipeline —
compilation, "measurement", analysis, evaluation, QPU update — runs without any hardware attached.

| Layer | Experiment | Depends on | Calibrated parameter |
|-------|-----------|------------|----------------------|
| `rabi` | Amplitude Rabi | — | π-pulse amplitude |
| `ramsey` | Ramsey | `rabi` | Drive frequency |
| `rabi_fine` | Amplitude Fine (x180) | `ramsey` | π-pulse amplitude (refined) |
| `drag` | DRAG quadrature scaling | `rabi_fine` | DRAG coefficient |
| `rabi_fine_2` | Amplitude Fine (x90) | `drag` | x90 amplitude |
| `sqrb` | Single-qubit RB | `rabi_fine_2` | Gate fidelity benchmark |

### 4.1 Load the hardware setup and QPU

Both are pre-serialized to JSON, so there is no descriptor or YAML to write by hand — `serializers.load`
reconstructs the `DeviceSetup` (an SHFQC, an HDAWG and a PQSC) and the six-qubit `QPU` exactly as they were
saved, starting calibration included.

In [ ]:
from laboneq import serializers, workflow

setup = serializers.load("automation_helpers/setup_synthetic_6q.json")
qpu = serializers.load("automation_helpers/qpu_synthetic_6q.json")

session = Session(device_setup=setup)
session.connect(do_emulation=True)

logbook = workflow.logbook.FolderStore("./laboneq_output")
logbook.activate()

print(f"{len(qpu.quantum_elements)} quantum elements, "
      f"{len(list(qpu.topology.edges()))} topology edges")

A quick look at the chip: the topology plot shows every qubit on the QPU (no couplers on this one).

In [ ]:
qpu.topology.plot(figsize=(6, 3), disconnected=True);

### 4.2 Pick three qubits

The result files cover all six qubits. For this lecture we pick just **three** of them — any subset
works, since injection only requires our subset's handles to be *among* the recorded ones.

In [ ]:
print("qubits on the chip:", [q.uid for q in qpu.quantum_elements])

qubit_ids = ["q1", "q3", "q5"]
print("qubits for this lecture:", qubit_ids)

### 4.3 Load the automation parameters

Sweep ranges, evaluation thresholds, and each layer's `inject_results` paths live in a YAML file, keyed
by layer and then by qubit, for all six qubits of the chip.

In [ ]:
import yaml

with open("automation_helpers/automation_params.yml") as f:
    automation_parameters = yaml.safe_load(f)

# e.g. the rabi layer's injection settings:
automation_parameters["rabi"]["options"]["inject_results"]

### 4.4 Demo switches

A few switches make the pipeline easier to demo or debug — set here once, then applied to every layer's
parameters below.

- `INJECT_MODE` controls when `inject_results` actually substitutes data (kept at `"emulation"`, matching
  the YAML default).
- `FORCE_PASS` / `SKIP_EVALUATION` / `SKIP_ANALYSIS` are debugging aids, left off below.

In [ ]:
FORCE_PASS      = False
SKIP_EVALUATION = False
SKIP_ANALYSIS   = False
INJECT_MODE     = "emulation"   # "emulation" | "always" | "no"

for layer_params in automation_parameters.values():
    eval_params = layer_params.get("evaluation_parameters", {})
    opts = layer_params.get("options", {})

    if FORCE_PASS:
        for key in eval_params:
            if "fit_r2_threshold" in key:
                eval_params[key] = {q: 0.0 for q in eval_params[key]}
            if "parameter_threshold" in key:
                eval_params[key] = {q: 1e6 for q in eval_params[key]}
    if SKIP_EVALUATION:
        opts["evaluate"] = False
    if SKIP_ANALYSIS:
        opts["do_analysis"] = False
    if "inject_results" in opts:
        opts["inject_results"]["use"] = INJECT_MODE

### 4.5 Build the automation graph

Six `WorkflowLayer`s (the same class from section 2), each depending on the previous one, mirror the
table above exactly — this is the same officially-supported `WorkflowAutomation` / `WorkflowLayer` API,
just with six real experiment workflows instead of one:

In [ ]:
from laboneq_applications.experiments import amplitude_rabi, ramsey, drag_q_scaling
from laboneq_applications.contrib.experiments import single_qubit_randomized_benchmarking

auto = WorkflowAutomation(
    session, qpu=qpu, parameters=automation_parameters, name="Q3_1Q_Bringup",
)

rabi_layer = WorkflowLayer(
    amplitude_rabi.experiment_workflow, qubit_ids,
    key="rabi", depends_on={"root"}, sequential=False,
)
ramsey_layer = WorkflowLayer(
    ramsey.experiment_workflow, qubit_ids,
    key="ramsey", depends_on={"rabi"}, sequential=False,
)
rabi_fine_layer = WorkflowLayer(
    amplitude_rabi.experiment_workflow, qubit_ids,
    key="rabi_fine", depends_on={"ramsey"}, sequential=False,
)
drag_layer = WorkflowLayer(
    drag_q_scaling.experiment_workflow, qubit_ids,
    key="drag", depends_on={"rabi_fine"}, sequential=False,
)
rabi_fine_layer_2 = WorkflowLayer(
    amplitude_rabi.experiment_workflow, qubit_ids,
    key="rabi_fine_2", depends_on={"drag"}, sequential=False,
)
sqrb_layer = WorkflowLayer(
    single_qubit_randomized_benchmarking.experiment_workflow, qubit_ids,
    key="sqrb", depends_on={"rabi_fine_2"}, sequential=True,  # one qubit at a time
)

for layer in [rabi_layer, ramsey_layer, rabi_fine_layer, drag_layer, rabi_fine_layer_2, sqrb_layer]:
    auto.add_layer(layer)

### 4.6 Monitoring (optional)

The same `start_web_viewer` from section 1 — it starts a small local web server in a background thread
and returns immediately, so it can be left running while the pipeline executes below.

- **Web viewer** (port 5005): the calibration graph, node-by-node status, live as the pipeline runs

In [ ]:
from laboneq.automation.web_viewer.server import start_web_viewer

start_web_viewer(auto, port=5005)

### 4.7 Run the pipeline

This runs all six layers, in dependency order, for all three qubits. Thanks to the injected results this
completes in a few seconds, with no hardware attached.

In [ ]:
auto.run()

for layer_key in ["rabi", "ramsey", "rabi_fine", "drag", "rabi_fine_2", "sqrb"]:
    layer = auto[layer_key]
    print(f"{layer_key:<12} status={layer.status:<8} pass={layer.pass_count} fail={layer.fail_count}")

### 4.8 The graph, and the calibrated qubits

`auto.plot()` gives the same graph the web viewer shows, as a static figure — every layer here should be
green (`passed`).

In [ ]:
auto.plot(graph_type="layers", figsize=(4, 8));

And the payoff: every qubit's calibration parameters have been updated in place by the pipeline, from
whatever they were before to the values the injected Rabi/Ramsey/DRAG/RB data implies:

In [ ]:
for q in qubit_ids:
    p = qpu[q].parameters
    print(
        f"{q}:  pi-amplitude={p.ge_drive_amplitude_pi:.4f}  "
        f"ge frequency={p.resonance_frequency_ge / 1e9:.6f} GHz"
    )

### 4.9 When a qubit fails — and how to recover automatically

Real calibration runs don't always evaluate cleanly on every qubit. To see this — and the fix — we reuse
**the exact same `auto` automation** built in section 4.5, changing only one parameter: set `q5`'s Rabi
`fit_r2_threshold` to `1.0`, a bar no fit can reach (the Rabi fits on the injected data land around
r² ≈ 0.998). `auto.reset()` clears every layer's status back to `ready` without touching the graph
or the (now-broken) parameters, so `auto.run()` re-executes the whole six-layer pipeline from scratch.

In [ ]:
auto["rabi"].parameters["evaluation_parameters"]["fit_r2_thresholds"]["q5"] = 1.0

auto.reset()
auto.run()

for layer_key in ["rabi", "ramsey", "rabi_fine", "drag", "rabi_fine_2", "sqrb"]:
    layer = auto[layer_key]
    per_qubit = {q: str(auto.get_node(f"{layer_key}_{q}").status) for q in qubit_ids}
    print(f"{layer_key:<12} layer_status={str(layer.status):<8} {per_qubit}")

`rabi` reports `failed` — one of its *active* nodes (`q5`) failed, even though `q1`/`q3` passed. Every
downstream layer then shows `q5` as `deactivated`: with no retry logic attached, a permanently failed node
gets its descendants deactivated automatically (`q1`/`q3` sail through the rest of the pipeline
unaffected). This is exactly the situation `AutomationLogic` (section 1) is for: attach a
`FixedParameterUpdate` to the `rabi` layer that lowers `q5`'s threshold by a fixed amount on every failed
attempt, then reset (which also reactivates the deactivated descendants) and run the *same* `auto` again.

In [ ]:
from laboneq.automation.logic import FixedParameterUpdate

auto["rabi"].logic = FixedParameterUpdate(
    new_layer_key="rabi",
    parameter_changes={"evaluation_parameters": {"fit_r2_thresholds": {"q5": -0.001}}},
)

auto.reset()
auto.run()

for layer_key in ["rabi", "ramsey", "rabi_fine", "drag", "rabi_fine_2", "sqrb"]:
    layer = auto[layer_key]
    per_qubit = {q: str(auto.get_node(f"{layer_key}_{q}").status) for q in qubit_ids}
    print(f"{layer_key:<12} layer_status={str(layer.status):<8} {per_qubit}")

print("\nrabi fail_count per qubit:", auto["rabi"].fail_count)
print("q5's fit_r2_threshold after retries:",
      auto["rabi"].parameters["evaluation_parameters"]["fit_r2_thresholds"]["q5"])

Every layer now reports `passed` for all three qubits — `q5`'s Rabi retried (`1.0 → 0.999 → 0.998 …`)
until the threshold dropped below its actual r², and the rest of the pipeline picked back up for it
automatically. `fail_count` shows exactly how many attempts that took — the same mechanism as the toy
example in section 1, just wired to a real evaluation parameter on the same automation graph instead of a
brand-new one. `parameter_changes` can target any nested key under `workflow_parameters`, `options`, or
`evaluation_parameters` — sweep ranges, shot counts, or thresholds — the mechanism is identical.

## 5. Saving the updated QPU

After a successful run the QPU holds updated calibration parameters. Serializing it gives a snapshot that
can be loaded back in for the next session — e.g. as the starting point for the *next* automated
calibration run, or to feed into a real experiment notebook.

In [ ]:
save_qpu = False  # set True to actually write the file

if save_qpu:
    from datetime import datetime
    import os

    os.makedirs("qpu_saves", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    save_path = f"qpu_saves/{timestamp}_qpu_parameters.json"
    serializers.save(qpu, save_path)
    print(f"Saved: {save_path}")